In [1]:
from pyiceberg.catalog import load_catalog
import duckdb

PROJECT = "/Users/Shared/academy/learner/projects/ethereum_iceberg"
catalog = load_catalog(
    "local",
    type="sql",
    uri=f"sqlite:///{PROJECT}/catalog/pyiceberg_catalog.db",
    warehouse=f"file://{PROJECT}/warehouse",
)
S3 = "s3://aws-public-blockchain/v1.0/eth"
con = duckdb.connect()
con.execute("set s3_region = 'us-east-2'")
catalog.list_namespaces()

[('ref',), ('raw',)]

In [2]:
from pyiceberg.types import IntegerType, LongType, StringType
blocks = catalog.load_table("raw.blocks")
print(blocks.schema())

table {
  1: difficulty: optional double
  2: hash: optional string
  3: miner: optional string
  4: nonce: optional string
  5: number: optional long
  6: size: optional long
  7: timestamp: optional timestamp
  8: base_fee_per_gas: optional long
  9: gas_limit: optional long
  10: gas_used: optional long
  11: extra_data: optional string
  12: parent_hash: optional string
  13: state_root: optional string
  14: receipts_root: optional string
  15: transactions_root: optional string
  16: sha3_uncles: optional string
  17: transaction_count: optional long
  18: date: optional date
}


In [3]:
with blocks.update_schema() as update:
    update.rename_column("miner", "fee_recipient")
    update.add_column("chain", StringType(), doc="network name")
    update.add_column("blob_count", IntegerType())
    update.delete_column("sha3_uncles")
print(blocks.schema())

table {
  1: difficulty: optional double
  2: hash: optional string
  3: fee_recipient: optional string
  4: nonce: optional string
  5: number: optional long
  6: size: optional long
  7: timestamp: optional timestamp
  8: base_fee_per_gas: optional long
  9: gas_limit: optional long
  10: gas_used: optional long
  11: extra_data: optional string
  12: parent_hash: optional string
  13: state_root: optional string
  14: receipts_root: optional string
  15: transactions_root: optional string
  17: transaction_count: optional long
  18: date: optional date
  19: chain: optional string (network name)
  20: blob_count: optional int
}


In [4]:
with blocks.update_schema() as update:
    update.update_column("blob_count", field_type=LongType())
blocks.schema().find_field("blob_count")

NestedField(field_id=20, name='blob_count', field_type=LongType(), required=False)

In [5]:
blocks.scan(limit=3, selected_fields=("number", "fee_recipient", "chain", "blob_count")).to_pandas()

,fee_recipient,number,chain,blob_count
0,0x4838b106fce9647bdf1e7877bf73ce8b0bad5f97,26325867,None,NaN
1,0x4838b106fce9647bdf1e7877bf73ce8b0bad5f97,26325652,None,NaN
2,0xdadb0d80178819f2319190d340ce9a924f783711,26324915,None,NaN


In [6]:
first = blocks.snapshots()[0]
blocks.scan(snapshot_id=first.snapshot_id, limit=3, selected_fields=("number", "miner", "sha3_uncles")).to_pandas()

,miner,number,sha3_uncles
0,0xd87f3d6c5624e8b02be13c2c92f8511b88b94d96,25660274,0x1dcc4de8dec75d7aab85b567b6ccd41ad312451b948a...
1,0x396343362be2a4da1ce0c1c210945346fb82aa49,25662164,0x1dcc4de8dec75d7aab85b567b6ccd41ad312451b948a...
2,0x396343362be2a4da1ce0c1c210945346fb82aa49,25662675,0x1dcc4de8dec75d7aab85b567b6ccd41ad312451b948a...
